# Financial Fraud Detection - Exploratory Data Analysis

This notebook performs initial exploration of the PaySim financial transactions dataset to understand fraud patterns and data characteristics.

## Load Dataset

Load the PaySim transaction dataset into a Pandas DataFrame for analysis.

In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

print("Java:", os.environ["JAVA_HOME"])
print("Hadoop:", os.environ["HADOOP_HOME"])

Java: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot
Hadoop: C:\hadoop


In [2]:
import sys

print(sys.executable)

c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Scripts\python.exe


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [4]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
csv_path = data_dir / "PS_20174392719_1491204439457_log.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print("CSV path:", csv_path)

Dataset loaded successfully
CSV path: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\PS_20174392719_1491204439457_log.csv


## Preview Dataset

Display the first few records to understand the structure and available features.

In [5]:
spark.version

'3.5.9'

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [7]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
csv_path = data_dir / "PS_20174392719_1491204439457_log.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print("CSV path:", csv_path)

Dataset loaded successfully
CSV path: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\PS_20174392719_1491204439457_log.csv


In [8]:
fraud_df.count()

6362620

In [9]:
fraud_df.show(5)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

# Check Dataset Shape

In [10]:
print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

Rows: 6362620
Columns: 11


# Check Schema

In [11]:
fraud_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



## 1. Dataset Overview

This section provides a basic understanding of the financial transaction dataset.
We will analyze the number of records, available features, and data distribution
using PySpark DataFrame operations.

In [12]:
# Total records and columns

print("Total Transactions:", fraud_df.count())
print("Total Features:", len(fraud_df.columns))

Total Transactions: 6362620
Total Features: 11


In [13]:
fraud_df.show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

## 2. Sample Transaction Records

Displaying sample transactions to understand the structure and values of the dataset.

In [14]:
fraud_df.show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

## 3. Missing Value Analysis

Checking missing values in each column to ensure data quality before further analysis.

In [15]:
from pyspark.sql.functions import col, sum

missing_values = fraud_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in fraud_df.columns
    ]
)

missing_values.show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



## 4. Transaction Type Analysis

Analyzing different transaction categories to understand customer transaction behavior.

In [16]:
transaction_type = fraud_df.groupBy("type") \
    .count() \
    .orderBy(col("count").desc())

transaction_type.show()

+--------+-------+
|    type|  count|
+--------+-------+
|CASH_OUT|2237500|
| PAYMENT|2151495|
| CASH_IN|1399284|
|TRANSFER| 532909|
|   DEBIT|  41432|
+--------+-------+



## 5. Fraud Transaction Distribution

The target variable `isFraud` indicates whether a transaction is fraudulent.
This analysis helps identify the imbalance between normal and fraudulent transactions.

In [17]:
fraud_distribution = fraud_df.groupBy("isFraud") \
    .count()

fraud_distribution.show()

+-------+-------+
|isFraud|  count|
+-------+-------+
|      0|6354407|
|      1|   8213|
+-------+-------+



## 6. Fraud Percentage

Calculating the percentage of fraudulent transactions in the complete dataset.

In [18]:
total_transactions = fraud_df.count()

fraud_count = fraud_df.filter(
    col("isFraud") == 1
).count()

fraud_percentage = (fraud_count / total_transactions) * 100

print("Total Transactions:", total_transactions)
print("Fraud Transactions:", fraud_count)
print("Fraud Percentage:", fraud_percentage)

Total Transactions: 6362620
Fraud Transactions: 8213
Fraud Percentage: 0.12908204481801522


## 7. Transaction Amount Analysis

Analyzing transaction amounts to identify patterns between fraudulent and normal transactions.

In [19]:
fraud_df.select(
    "amount"
).describe().show()

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|           6362620|
|   mean|179861.90354913412|
| stddev| 603858.2314629498|
|    min|               0.0|
|    max|     9.244551664E7|
+-------+------------------+



## 8. Average Amount Comparison

Comparing average transaction amounts between fraudulent and legitimate transactions.

In [20]:
fraud_df.groupBy("isFraud") \
    .avg("amount") \
    .show()

+-------+------------------+
|isFraud|       avg(amount)|
+-------+------------------+
|      0| 178197.0417274114|
|      1|1467967.2991403837|
+-------+------------------+



## 9. Fraud Pattern by Transaction Type

Identifying which transaction types contain higher fraud activity.

In [21]:
fraud_by_type = fraud_df.groupBy("type","isFraud") \
    .count() \
    .orderBy("type")

fraud_by_type.show()

+--------+-------+-------+
|    type|isFraud|  count|
+--------+-------+-------+
| CASH_IN|      0|1399284|
|CASH_OUT|      1|   4116|
|CASH_OUT|      0|2233384|
|   DEBIT|      0|  41432|
| PAYMENT|      0|2151495|
|TRANSFER|      0| 528812|
|TRANSFER|      1|   4097|
+--------+-------+-------+



## 10. Feature Correlation Analysis

Checking relationships between numerical features to identify important variables
for fraud detection modeling.

In [22]:
numeric_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for column in numeric_columns:
    correlation = fraud_df.stat.corr(column,"isFraud")
    print(column, ":", correlation)

amount : 0.07668842884028321
oldbalanceOrg : 0.010154421850332844
newbalanceOrig : -0.008148161267570215
oldbalanceDest : -0.005885278228051785
newbalanceDest : 0.0005353470683179229


In [23]:
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")

print("Parquet configuration updated")

Parquet configuration updated


In [24]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

HADOOP_HOME: C:\hadoop


In [25]:
spark.stop()

In [26]:
from pyspark.sql import SparkSession
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [31]:
spark.stop()


## 11. Saving Processed Dataset

Saving the Spark DataFrame in Parquet format for faster future processing.

In [32]:
from pathlib import Path

output_path = project_root / "data" / "fraud_spark_processed.csv"

fraud_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(output_path))

print("CSV saved successfully:", output_path)

Py4JJavaError: An error occurred while calling o191.csv.
: java.util.NoSuchElementException: None.get
	at scala.None$.get(Option.scala:529)
	at scala.None$.get(Option.scala:527)
	at org.apache.spark.sql.execution.datasources.BasicWriteJobStatsTracker$.metrics(BasicWriteStatsTracker.scala:239)
	at org.apache.spark.sql.execution.command.DataWritingCommand.metrics(DataWritingCommand.scala:55)
	at org.apache.spark.sql.execution.command.DataWritingCommand.metrics$(DataWritingCommand.scala:55)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.metrics$lzycompute(InsertIntoHadoopFsRelationCommand.scala:47)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.metrics(InsertIntoHadoopFsRelationCommand.scala:47)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.metrics$lzycompute(commands.scala:109)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.metrics(commands.scala:109)
	at org.apache.spark.

## Conclusion

The Spark-based EDA identified important characteristics of financial transactions,
including fraud distribution, transaction type patterns, amount behavior, and
important numerical features. These insights will be used for feature engineering
and fraud detection model development.